In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import joblib
import pandas as pd
import seaborn as sns
from   sklearn.pipeline import Pipeline
import optuna
import shap
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix
from catboost import CatBoostClassifier
from sklearn.metrics import make_scorer, classification_report

np.random.seed(42)


# Läs in träningsfilen
df = pd.read_csv(r"C:\Users\Eier\OneDrive\Dokument\DataScience\Data_science_projektkurs\archive (1)\diabetes_prediction_dataset.csv")
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df['gender'].unique())
print(df['smoking_history'].unique())
df_encoded = pd.get_dummies(df, columns=['gender', 'smoking_history'], drop_first=False)
print(df_encoded.head())
print(df_encoded.columns.tolist())
numeric_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 
                'HbA1c_level', 'blood_glucose_level', 'diabetes']

df_numeric = df_encoded[numeric_cols]
print(df_numeric.describe().T)
print(df['diabetes'].value_counts())

Y = df_encoded['diabetes']
X = df_encoded.drop(columns=['diabetes'])

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB
None
                 age  hypertension  heart_disease            bmi  \
count  100000.000000  100000.00000  100000.000000  100000.000000   
mean       41.885856       0.07485       0.039420      27.320767   
std        22.516840       0.26315       0.194593       6.636783   
min   

In [4]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=Y
)

# 15% val, 15% test
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42, stratify=Y_temp
)

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)
print(X_test.shape, Y_test.shape)

(70000, 15) (70000,)
(15000, 15) (15000,)
(15000, 15) (15000,)


In [12]:

cb = Pipeline([
    ('model', CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        random_state=42,
        verbose=False
    ))
])

cb.fit(X_train, Y_train)
score = cb.score(X_val, Y_val)

print("Modellen är tränad")
print("CatBoostClassifier:", score)

joblib.dump(cb, 'cb_trained.pkl')

Modellen är tränad
CatBoostClassifier: 0.9726428571428571


['cb_trained.pkl']

In [13]:
# Ladda modellen
cb_pipeline = joblib.load('cb_trained.pkl')

# Prediktioner
y_pred = cb_pipeline.predict(X_test)

# Rapport
print("Classification Report för CatBoostClassifier")
print(classification_report(Y_test, y_pred))

Classification Report för CatBoostClassifier
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     13725
           1       0.97      0.68      0.80      1275

    accuracy                           0.97     15000
   macro avg       0.97      0.84      0.89     15000
weighted avg       0.97      0.97      0.97     15000



In [14]:
# Ladda den tränade modellen
cb = joblib.load('cb_trained.pkl') 

# Gör prediktioner på valideringsdata med den optimerade modellen
Y_val_pred_opt = cb.predict(X_val)

# Utvärdera modellens prestanda
accuracy_opt = accuracy_score(Y_val, Y_val_pred_opt)
precision_opt = precision_score(Y_val, Y_val_pred_opt, average='macro')
recall_opt = recall_score(Y_val, Y_val_pred_opt, average='macro')
f1_opt = f1_score(Y_val, Y_val_pred_opt, average='weighted')

# Samla resultaten
cb_results = [{
    'Model': 'CatBoostClassifier ',
    'Accuracy': accuracy_opt,
    'Precision': precision_opt,
    'Recall': recall_opt,
    'F1-score': f1_opt
}]

# Spara resultaten
joblib.dump(cb_results, 'cb_results.pkl')

# Konvertera listan till en DataFrame och visa resultaten
cb_results = pd.DataFrame(cb_results)
cb_results

,Model,Accuracy,Precision,Recall,F1-score
0,CatBoostClassifier,0.972643,0.963267,0.848923,0.970721


Hyperparameroptimering för modellen CatBooster

In [8]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# -------------------------------------------------------
# Convergence stopper
# -------------------------------------------------------
class ConvergenceStopper:
    def __init__(self, threshold=0.002, patience=5):
        self.threshold = threshold
        self.patience = patience
        self.last_best = None
        self.bad_count = 0

    def __call__(self, study, trial):
        current = trial.value
        best = study.best_value

        if self.last_best is None:
            self.last_best = best
            return

        improvement = best - self.last_best

        if improvement < self.threshold:
            self.bad_count += 1
        else:
            self.bad_count = 0
            self.last_best = best

        if self.bad_count >= self.patience:
            print(f"\n[OPTUNA] {self.patience} dåliga trials i rad → stoppar studien.")
            study.stop()

# -------------------------------------------------------
# CatBoost hyperparameter space
# -------------------------------------------------------
def cb_param_space(trial):
    return {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 5.0),
        "iterations": trial.suggest_int("iterations", 300, 800),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
    }

# -------------------------------------------------------
# Objective-funktion
# -------------------------------------------------------
def objective_cb(trial):
    params = cb_param_space(trial)

    model = CatBoostClassifier(
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["l2_leaf_reg"],
        iterations=params["iterations"],
        border_count=params["border_count"],
        bagging_temperature=params["bagging_temperature"],
        loss_function="Logloss",
        eval_metric="F1",
        random_seed=535,
        verbose=False
    )

    model.fit(X_train, Y_train)
    preds = model.predict(X_val)

    score = f1_score(Y_val, preds)

    trial.set_user_attr("best_model", model)

    return score

# -------------------------------------------------------
# Train/val split
# -------------------------------------------------------
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.2, random_state=535
)

# -------------------------------------------------------
# Kör Optuna-studien (Steg 1)
# -------------------------------------------------------
stopper = ConvergenceStopper(threshold=0.002, patience=5)

study_cb = optuna.create_study(direction="maximize")
study_cb.optimize(objective_cb, n_trials=100, callbacks=[stopper])

# -------------------------------------------------------
# Spara bästa modellen
# -------------------------------------------------------
best_cb = study_cb.best_trial.user_attrs["best_model"]
joblib.dump(best_cb, "best_cb_model.joblib")

print("\n[OPTUNA] Bästa CatBoost-parametrar:")
print(study_cb.best_params)
print("[JOBLIB] Sparad som best_cb_model.joblib")

[I 2026-10-04 18:57:43,424] A new study created in memory with name: no-name-9e501164-f510-4310-a230-26677d4b3483
[I 2026-10-04 18:57:46,507] Trial 0 finished with value: 0.8057913130304544 and parameters: {'depth': 7, 'learning_rate': 0.13225568084048694, 'l2_leaf_reg': 4.831056134221898, 'iterations': 388, 'border_count': 100, 'bagging_temperature': 0.9030351848622471}. Best is trial 0 with value: 0.8057913130304544.
[I 2026-10-04 18:57:48,676] Trial 1 finished with value: 0.8101010101010101 and parameters: {'depth': 5, 'learning_rate': 0.08121933841414587, 'l2_leaf_reg': 4.973064201864462, 'iterations': 328, 'border_count': 195, 'bagging_temperature': 0.06261041410058665}. Best is trial 1 with value: 0.8101010101010101.
[I 2026-10-04 18:57:51,975] Trial 2 finished with value: 0.8097165991902834 and parameters: {'depth': 6, 'learning_rate': 0.03533245397324101, 'l2_leaf_reg': 1.7951788451588864, 'iterations': 418, 'border_count': 190, 'bagging_temperature': 0.08008854232990537}. Best


[OPTUNA] 5 dåliga trials i rad → stoppar studien.

[OPTUNA] Bästa CatBoost-parametrar:
{'depth': 6, 'learning_rate': 0.025638830849452704, 'l2_leaf_reg': 2.164919548127638, 'iterations': 455, 'border_count': 194, 'bagging_temperature': 0.4836424273064852}
[JOBLIB] Sparad som best_cb_model.joblib


In [11]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# -------------------------------------------------------
# Ladda modellen från joblib
# -------------------------------------------------------
hgb = joblib.load("best_cb_model.joblib")
print("[JOBLIB] CB-modellen laddad.")

# -------------------------------------------------------
# Prediktioner
# -------------------------------------------------------
preds = hgb.predict(X_test)
probs = hgb.predict_proba(X_test)[:, 1]

# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
acc = accuracy_score(Y_test, preds)
f1 = f1_score(Y_test, preds)
prec = precision_score(Y_test, preds)
rec = recall_score(Y_test, preds)
auc = roc_auc_score(Y_test, probs)
cm = confusion_matrix(Y_test, preds)

# -------------------------------------------------------
# Utskrifter
# -------------------------------------------------------
print("\n====================")
print("   CB– METRICS")
print("====================")

print(f"Accuracy:      {acc:.4f}")
print(f"F1-score:      {f1:.4f}")
print(f"Precision:     {prec:.4f}")
print(f"Recall:        {rec:.4f}")
print(f"ROC-AUC:       {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(Y_test, preds))


[JOBLIB] CB-modellen laddad.

   CB– METRICS
Accuracy:      0.9711
F1-score:      0.7994
Precision:     0.9762
Recall:        0.6769
ROC-AUC:       0.9787

Confusion Matrix:
[[13704    21]
 [  412   863]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     13725
           1       0.98      0.68      0.80      1275

    accuracy                           0.97     15000
   macro avg       0.97      0.84      0.89     15000
weighted avg       0.97      0.97      0.97     15000



In [20]:
# Ladda den sparade modellen
cb = joblib.load('best_cb_model.joblib')  # Ändra filnamnet till den sparade modellen

# Definiera scorer
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro')
}

# Beräkna cross-validation scores
cv_results = cross_validate(cb, X_train, Y_train, cv=5, scoring=scoring)

# Skapa en lista med resultaten
cb_results = []
for fold in range(5):
    cb_results.append({
        'Fold': fold,
        'Accuracy': cv_results['test_accuracy'][fold],
        'Precision': cv_results['test_precision'][fold],
        'Recall': cv_results['test_recall'][fold],
        'F1-score': cv_results['test_f1'][fold]
    })

# Konvertera listan till en DataFrame
cb_results = pd.DataFrame(cb_results)
print(cb_results)

# Spara DataFrame med joblib

   Fold  Accuracy  Precision    Recall  F1-score
0     0  0.971339   0.967177  0.845244  0.895168
1     1  0.973571   0.982472  0.848351  0.902408
2     2  0.971875   0.975316  0.842709  0.896085
3     3  0.970982   0.973338  0.838281  0.892300
4     4  0.968929   0.970474  0.827246  0.883422
